---
## Data Cleaning Script Template
### Student Name: Nicole Riley
### Dataset: CERT Error Rate Data from FY 2021-2025
---

In [1]:
# Import libraries - 
# pandas will be able to do most of my CSV cleaning tasks

import pandas as pd

I want to be able to compare multiple years of data, so I will load in data sets for fiscal years 2021-2025

In [2]:
# Loading data files -

df_2021 = pd.read_csv('Medicare_FFS_CERT_2021.csv')
df_2022 = pd.read_csv('Medicare_FFS_CERT_2022.csv')
df_2023 = pd.read_csv('Medicare_FFS_CERT_2023.csv')
df_2024 = pd.read_csv('Medicare_FFS_CERT_2024.csv')
df_2025 = pd.read_csv('2025_CERT_Public_Data.csv')

## Explore the Data

I'll first take a look at the a few of the primary features of the 2025 data set, including the number of rows and columns, the column names, and how many columns contain null values.

In [3]:
print(df_2025.shape)
print(df_2025.info())
df_2025.sample(20)

(163940, 8)
<class 'pandas.DataFrame'>
RangeIndex: 163940 entries, 0 to 163939
Data columns (total 8 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   claim_control_number  163940 non-null  int64  
 1   Part                  163940 non-null  str    
 2   DRG                   8750 non-null    float64
 3   HCPCS Procedure Code  117082 non-null  str    
 4   Provider Type         163940 non-null  str    
 5   Type of Bill          119682 non-null  str    
 6   Review Decision       163940 non-null  str    
 7   Error Code            163940 non-null  str    
dtypes: float64(1), int64(1), str(6)
memory usage: 10.0 MB
None


,claim_control_number,Part,DRG,HCPCS Procedure Code,Provider Type,Type of Bill,Review Decision,Error Code
46952,2542218,3. Part A(Excluding Inpatient Hospital PPS),NaN,NaN,Non PPS Short Term Hospital Inpatient,111,Agree,-
154258,2568999,3. Part A(Excluding Inpatient Hospital PPS),NaN,NaN,"OPPS, Laboratory (an FI), Ambulatory (Billing ...",131,Agree,-
86212,2552180,3. Part A(Excluding Inpatient Hospital PPS),NaN,J1650,Inpatient Critical Access Hospital,110,Agree,-
155114,2559065,3. Part A(Excluding Inpatient Hospital PPS),NaN,83970,ESRD,721,Agree,-
7061,2533012,1. Part B,NaN,98981,Nurse Practitioner,NaN,Agree,-
156711,2565336,1. Part B,NaN,87481,Clinical Laboratory (Billing Independently),NaN,Disagree,No Documentation
51525,2542100,3. Part A(Excluding Inpatient Hospital PPS),NaN,G0300,HHA,329,Disagree,Incorrect Coding
2182,2533791,3. Part A(Excluding Inpatient Hospital PPS),NaN,83735,"OPPS, Laboratory (an FI), Ambulatory (Billing ...",121,Agree,-
74311,2549543,3. Part A(Excluding Inpatient Hospital PPS),NaN,NaN,SNF,214,Disagree,Insufficient Documentation
115753,2559532,3. Part A(Excluding Inpatient Hospital PPS),NaN,NaN,SNF,214,Agree,-


I see some interesting things emerging. It appears that the DRG column contains quite a few null values, with only 8,750 of the dataframe's almost 164,000 rows containing data. I also see that the Type of Bill column is missing about 50,000 values, as is the HCPCS procedure code column. I'll want to look into this further.

First, though, I know I want to be able to compare this data across the five years I've loaded. I believe the best way to do this is to combine them into a single dataframe with a new column indicating the fiscal year. Before I can do that, I need to make sure that the data is consistent across all five files.

In [4]:
# I want make sure that all of my dataframes have the same columns
# I'll use a for loop to print the columns from each dataframe to a list so I can see them all in one place, lined up
# if they all look the same, I should be able to combine them without needing to do much more cleaning

for df in [df_2021, df_2022, df_2023, df_2024, df_2025]:
    print(df.columns.tolist())

['claim_control_number', 'Part', 'DRG', 'HCPCS Procedure Code', 'Provider Type', 'Type of Bill', 'Review Decision', 'Error Code']
['claim_control_number', 'Part', 'DRG', 'HCPCS Procedure Code', 'Provider Type', 'Type of Bill', 'Review Decision', 'Error Code']
['claim_control_number', 'Part', 'DRG', 'HCPCS Procedure Code', 'Provider Type', 'Type of Bill', 'Review Decision', 'Error Code']
['claim_control_number', 'Part', 'DRG', 'HCPCS Procedure Code', 'Provider Type', 'Type of Bill', 'Review Decision', 'Error Code']
['claim_control_number', 'Part', 'DRG', 'HCPCS Procedure Code', 'Provider Type', 'Type of Bill', 'Review Decision', 'Error Code']


It looks consistent, so I shouldn't have any problems with the columns lining up appropriately. 

However, it does look like the claim control number is the only column header currently in snake case, with all other column headers in title case with spaces by default. I will change the claim control number column header to title case with spaces as well, for consistency and Power BI readability.

I also want to make sure all these columns contain the same type of data, and that the type of data makes sense for the contents, as well as for the kind of analysis I may want to perform.

In [5]:
# I'll use another for loop to print the data types in all of my dataframes
# I'll use the zip function to add a list of the corresponding years so I can refer to each dataframe by it's year for the purposes of displaying the results
for year, df in zip([2021, 2022, 2023, 2024, 2025], [df_2021, df_2022, df_2023, df_2024, df_2025]):
    print(f"\n{year}:")
    print(df.dtypes)


2021:
claim_control_number      int64
Part                        str
DRG                     float64
HCPCS Procedure Code        str
Provider Type               str
Type of Bill                str
Review Decision             str
Error Code                  str
dtype: object

2022:
claim_control_number      int64
Part                        str
DRG                     float64
HCPCS Procedure Code        str
Provider Type               str
Type of Bill                str
Review Decision             str
Error Code                  str
dtype: object

2023:
claim_control_number      int64
Part                        str
DRG                     float64
HCPCS Procedure Code        str
Provider Type               str
Type of Bill                str
Review Decision             str
Error Code                  str
dtype: object

2024:
claim_control_number      int64
Part                        str
DRG                     float64
HCPCS Procedure Code        str
Provider Type               str
Ty

It looks like the types are consistent across all five data sets, so I should not run into issues when I go to combine them.

One thing I notice is that the DRG column contains float values. I know that Medicare DRG numbers should be three-digit numbers. I'll investigate further by filtering one dataframe to only show data where the value in the DRG column is not null, then take a random sample.

In [6]:
df_2025[df_2025['DRG'].notnull()].sample(25)

,claim_control_number,Part,DRG,HCPCS Procedure Code,Provider Type,Type of Bill,Review Decision,Error Code
35507,2533035,4. Part A(Inpatient Hospital PPS),669.0,NaN,DRG Short Term,110,Agree,-
160672,2567189,4. Part A(Inpatient Hospital PPS),433.0,NaN,DRG Short Term,111,Agree,-
136088,2560544,4. Part A(Inpatient Hospital PPS),981.0,NaN,DRG Short Term,111,Agree,-
103182,2551850,4. Part A(Inpatient Hospital PPS),689.0,NaN,DRG Short Term,111,Agree,-
158281,2555534,4. Part A(Inpatient Hospital PPS),259.0,NaN,DRG Short Term,111,Agree,-
158733,2566689,4. Part A(Inpatient Hospital PPS),603.0,NaN,DRG Short Term,110,Agree,-
52953,2543883,4. Part A(Inpatient Hospital PPS),455.0,NaN,DRG Short Term,111,Disagree,Medical Necessity
38519,2540980,4. Part A(Inpatient Hospital PPS),454.0,NaN,DRG Short Term,111,Agree,-
160435,2566900,4. Part A(Inpatient Hospital PPS),561.0,NaN,DRG Short Term,111,Agree,-
106016,2553876,4. Part A(Inpatient Hospital PPS),948.0,NaN,DRG Short Term,111,Agree,Overturned


The DRG column presents an interesting challenge. It contains null values for the vast majority of records, as it is only relevant for inpatient hospital PPS claims. However, for that inpatient data the DRG information could offer valuable insight into error trends by diagnosis. 

Before dropping the column from the primary dataset, I will first create a separate inpatient subset containing only Part A/IPPS records where DRG data is present. This subset will retain the DRG column and serve as the basis for a focused inpatient analysis.

Once the subset has been created, the DRG column will be dropped from the primary combined dataset to keep it clean and avoid complications from a predominantly null column.

Upon reviewing the preview, I'm noticing another inconsistency I will want to address: the Part column contains a number before the description of the Medicare part. I want to see what these values look like.

In [7]:
# preview the contents of the Part column with .unique()

df_2025['Part'].unique()

<StringArray>
[                                 '2. DME MAC',
                                   '1. Part B',
 '3. Part A(Excluding Inpatient Hospital PPS)',
           '4. Part A(Inpatient Hospital PPS)']
Length: 4, dtype: str

I want to be able to use the Part values for sorting in Power BI, and the prefix numbers at the beginning of the string are not providing any additional meaning, so I will strip them out of the final file.

### Extra Context - DRG Reference File

Since I know I want to do additional analysis on the inpatient hospital / DRG data, I want to pull information about what the DRG numbers actually mean. The DRG reference terminology was sourced from the Tuva Project, an open-source healthcare data platform widely used by healthcare data professionals. 

*Note: MS-DRG descriptions may vary slightly across fiscal years. The Tuva reference file reflects current MS-DRG assignments and is applied uniformly across all years in this dataset.*

*See README for full source documentation and citation.* 

In [8]:
# I need to load in the DRG reference set from Tuva and preview it
DRG_ref = pd.read_csv('ms_drg.csv_0_0_0.csv')

DRG_ref.sample(15)

,001,\N,P,Heart Transplant or Implant of Heart Assist System with MCC,0,\N.1
202,263,05,P,Vein Ligation and Stripping,0,\N
129,173,04,P,Ultrasound Accelerated and Other Thrombolysis ...,0,\N
322,399,06,P,Appendix Procedures without CC/MCC,0,\N
641,809,16,M,Major Hematological and Immunological Diagnose...,0,\N
522,650,11,P,Kidney Transplant with Hemodialysis with MCC,0,\N
283,352,06,P,Inguinal and Femoral Hernia Procedures without...,0,\N
816,766,\N,\N,Cesarean section w/o CC/MCC,1,2018-10-01
307,383,06,M,Uncomplicated Peptic Ulcer with MCC,0,\N
592,741,13,P,Uterine and Adnexa Procedures for Non-Ovarian ...,0,\N
318,394,06,M,Other Digestive System Diagnoses with CC,0,\N


Upon loading in the Tuva DRG reference file, I encountered two issues that required additional handling.

First, the dataset does not contain column headings. The Tuva Project provides column documentation for this file on their website. Using this documentation, I was able to assign the following column names by repeating the loading process: ms_drg_code, mdc_code, medical_surgical, ms_drg_description, deprecated, and deprecated_date. I will convert these column headers to title case with spaces in order to be consistent with the primary data set.

Second, the file contains \N values in several columns. This is a MySQL-style null indicator commonly used when exporting CSV files from a MySQL database. Rather than leaving these as-is, I will replace them with NaN to maintain consistency with the null representation used throughout the rest of my dataset. I'll need numpy for this, so I will import this library now.

Finally, the last two columns contain information about DRG code depreciation. This is more complexity than is required for this exploratory dashboard, so these two columns will be dropped.

In [9]:
import numpy as np

### Based on the exploration above, the following cleaning and preparation steps will be performed:

- Add a fiscal_year column to each CERT dataset and concatenate into a single combined file
- Normalize the column headers in the combined file to title case with spaces
- Remove the prefix number from the Part column to improve readability and sortability
- Create a separate inpatient subset containing only Part A/IPPS claims where DRG data is present
- Cast the DRG column to string in the inpatient subset
- Clean the Tuva DRG reference file by adding normalized title case with spaced headers, replacing nulls, and dropping deprecation information columns
- Merge the cleaned Tuva DRG reference file with the inpatient subset for additional clinical context
- Drop the DRG column from the combined dataset
- Export both cleaned datasets as CSV files

---
## Data Cleaning Script
---

In [10]:
# add a Fiscal Year column for identification of the data year after union

for name,df in zip([2021,2022,2023,2024,2025],[df_2021,df_2022,df_2023,df_2024,df_2025]):
    df['Fiscal Year'] = name

In [11]:
# combine the files into a single CERT dataframe

full_CERT_df = pd.concat([df_2021,df_2022,df_2023,df_2024,df_2025])

In [12]:
# normalize column names to title case with spaces

full_CERT_df = full_CERT_df.rename(columns={'claim_control_number':'Claim Control Number'})

In [13]:
# remove numeric prefixes from Part column

full_CERT_df['Part'] = full_CERT_df['Part'].str.split('. ', n=1).str[1]

In [14]:
# create DRG subset dataframe

DRG_subset_df = full_CERT_df[full_CERT_df['DRG'].notnull()]

In [15]:
# in DRG subset, convert DRG column to integer for reference data set merge

DRG_subset_df['DRG'] = DRG_subset_df['DRG'].astype(int)

In [16]:
# reload Tuva reference data with column headers, normalized to title case with spaces

DRG_ref = pd.read_csv('ms_drg.csv_0_0_0.csv', header=None, names=['MS-DRG Code', 'MDC Code', 'Medical or Surgical', 'MS-DRG Description', 'deprecated' ,'deprecated_date'])

In [17]:
# drop the deprecated and deprecated date columns from reference df

DRG_ref = DRG_ref.drop(columns=['deprecated','deprecated_date'])

In [18]:
# replace special character with NaN

DRG_ref = DRG_ref.replace(r'\N', np.nan)

In [19]:
# merge cleaned Tuva data into the DRG subset

DRG_subset_df = pd.merge(DRG_subset_df,DRG_ref,left_on='DRG',right_on='MS-DRG Code',how='left')

In [20]:
# drop extra DRG column, and convert the reamaining DRG column to string type to ensure proper handling in Power BI

DRG_subset_df = DRG_subset_df.drop(columns=['DRG'])
DRG_subset_df['MS-DRG Code'] = DRG_subset_df['MS-DRG Code'].astype(str)

In [21]:
# drop DRG column from primary dataframe

full_CERT_df = full_CERT_df.drop(columns='DRG')

---
## Save the Cleaned Datasets
---

In [22]:
full_CERT_df.to_csv("cleaned_data_full_CERT.csv", index=False)

DRG_subset_df.to_csv("cleaned_data_DRG_sub.csv", index=False)

In [23]:
print("Data cleaning complete. Cleaned CSV files created.")

Data cleaning complete. Cleaned CSV files created.
